# MCMC Support

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markean/aimz/blob/main/docs/notebooks/mcmc.ipynb)

While aimz is primarily designed around variational inference and predictive sampling, it also provides support for MCMC methods via the [NumPyro backend](https://num.pyro.ai/en/stable/mcmc.html#numpyro.infer.mcmc.MCMC), using the same aimz interface (e.g., `.fit_on_batch()` and `.predict_on_batch()`). This enables users to apply MCMC to more complex models where variational inference may be less effective and dataset sizes are relatively small.

In [1]:
import logging

import jax.numpy as jnp
import numpyro.distributions as dist
from jax import random
from jax.typing import ArrayLike
from numpyro import plate, sample
from numpyro.infer import MCMC, NUTS

from aimz import ImpactModel

logging.basicConfig(level=logging.INFO)

%load_ext watermark

## Model and Data

We set up a linear regression model and create synthetic data for both features and targets as an example.

In [2]:
def model(X: ArrayLike, y: ArrayLike | None = None) -> None:
    """Linear regression model."""
    w = sample("w", dist.Normal().expand((X.shape[1],)))
    b = sample("b", dist.Normal())
    mu = jnp.dot(X, w) + b
    sigma = sample("sigma", dist.Exponential())
    with plate("obs", size=X.shape[0]):
        sample("y", dist.Normal(mu, sigma), obs=y)


rng_key = random.key(42)
rng_key, rng_key_w, rng_key_b, rng_key_x, rng_key_e = random.split(rng_key, 5)
w = random.normal(rng_key_w, (10,))
b = random.normal(rng_key_b)
X = random.normal(rng_key_x, (1000, 10))
e = random.normal(rng_key_e, (1000,))
y = jnp.dot(X, w) + b + e

INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


## MCMC Sampling and Prediction

MCMC sampling can be performed using the `ImpactModel` class by setting the `inference` argument to `MCMC`. Users can configure the sampler, warm-up steps, and other MCMC-specific parameters. Calling `.fit_on_batch()` initiates the sampling process. Internally, aimz executes the sampler via the `.run()` method and stores the posterior samples using `.get_samples()`.

Note that calling `.fit()` with `MCMC` as the inference method will raise a `TypeError`, as this method is intended for mini-batch training or subsampling. The `.posterior` property holds the samples combined across chains (`num_chains`). The returned output trees keep the chains along their `chain` dimension, so chain-aware diagnostics such as R-hat work on them directly. Posterior predictive sampling can be performed using the `.predict()` or `.predict_on_batch()` methods.

In [3]:
rng_key, rng_subkey = random.split(rng_key)
im = ImpactModel(
    model,
    rng_key=rng_subkey,
    inference=MCMC(NUTS(model), num_warmup=500, num_samples=500),
)
im.fit_on_batch(X, y)
im.inference.print_summary()
im.predict_on_batch(X)

INFO:aimz.model.impact_model:Backend: cpu, Devices: 1
INFO:aimz.model.impact_model:Drawing posterior samples (num_samples=500)
sample: 100%|██████████| 1000/1000 [00:01<00:00, 574.70it/s, 7 steps of size 6.87e-01. acc. prob=0.88]



                mean       std    median      5.0%     95.0%     n_eff     r_hat
         b      0.41      0.03      0.41      0.35      0.45    854.53      1.00
     sigma      1.03      0.02      1.03      1.00      1.07    912.45      1.00
      w[0]      0.59      0.03      0.59      0.54      0.65    940.88      1.00
      w[1]      0.86      0.03      0.86      0.81      0.92    997.07      1.00
      w[2]     -0.90      0.03     -0.90     -0.97     -0.86   1108.69      1.00
      w[3]     -0.60      0.03     -0.60     -0.66     -0.54    777.51      1.00
      w[4]     -1.24      0.03     -1.24     -1.29     -1.19   1110.16      1.00
      w[5]     -0.80      0.03     -0.80     -0.85     -0.75   1367.91      1.00
      w[6]     -0.51      0.03     -0.51     -0.57     -0.46   1120.50      1.00
      w[7]     -1.22      0.03     -1.22     -1.27     -1.16   1003.52      1.00
      w[8]     -0.17      0.03     -0.16     -0.22     -0.12    835.84      1.00
      w[9]     -0.10      0

<xarray.DataTree 'root'>
Group: /
├── Group: /posterior
│       Dimensions:  (chain: 1, draw: 500, w_dim_0: 10)
│       Coordinates:
│         * chain    (chain) int64 8B 0
│         * draw     (draw) int64 4kB 0 1 2 3 4 5 6 7 ... 493 494 495 496 497 498 499
│         * w_dim_0  (w_dim_0) int64 80B 0 1 2 3 4 5 6 7 8 9
│       Data variables:
│           b        (chain, draw) float32 2kB 0.4255 0.4507 0.3977 ... 0.3845 0.4487
│           sigma    (chain, draw) float32 2kB 1.02 1.01 1.032 ... 1.029 1.035 1.025
│           w        (chain, draw, w_dim_0) float32 20kB 0.6008 0.8825 ... -0.08485
│       Attributes:
│           created_at:    2026-11-16T19:27:55.814558+00:00
│           aimz_version:  0.18.0
└── Group: /posterior_predictive
        Dimensions:  (chain: 1, draw: 500, obs: 1000)
        Coordinates:
          * chain    (chain) int64 8B 0
          * draw     (draw) int64 4kB 0 1 2 3 4 5 6 7 ... 493 494 495 496 497 498 499
          * obs      (obs) int64 8kB 0 1 2 3 4 5 6 7 ... 992 993 994 995 996 997 998 999
        Data variables:
            y        (chain, draw, obs) float32 2MB 2.966 -1.517 ... -4.622 -1.448
        Attributes:
            created_at:    2026-11-16T19:27:55.816936+00:00
            aimz_version:  0.18.0
            rng_key:       [2477523575, 3040475525]

## Using External MCMC Samples

Users can run MCMC sampling directly using NumPyro and then insert the posterior samples into an `ImpactModel` instance using the `.set_posterior_sample()` method for downstream analysis.
For example:

In [4]:
mcmc = MCMC(NUTS(model), num_warmup=1000, num_samples=1000)
rng_key, rng_subkey = random.split(rng_key)
mcmc.run(rng_subkey, X, y)

im.set_posterior_sample(mcmc.get_samples())
im.predict_on_batch(X)

sample: 100%|██████████| 2000/2000 [00:02<00:00, 758.33it/s, 7 steps of size 6.32e-01. acc. prob=0.89] 


<xarray.DataTree 'root'>
Group: /
├── Group: /posterior
│       Dimensions:  (chain: 1, draw: 1000, w_dim_0: 10)
│       Coordinates:
│         * chain    (chain) int64 8B 0
│         * draw     (draw) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
│         * w_dim_0  (w_dim_0) int64 80B 0 1 2 3 4 5 6 7 8 9
│       Data variables:
│           b        (chain, draw) float32 4kB 0.4403 0.4313 0.3803 ... 0.4207 0.4113
│           sigma    (chain, draw) float32 4kB 1.04 1.044 1.027 ... 1.02 1.058 1.002
│           w        (chain, draw, w_dim_0) float32 40kB 0.624 0.8634 ... -0.0818
│       Attributes:
│           created_at:    2026-11-16T19:27:59.889579+00:00
│           aimz_version:  0.18.0
└── Group: /posterior_predictive
        Dimensions:  (chain: 1, draw: 1000, obs: 1000)
        Coordinates:
          * chain    (chain) int64 8B 0
          * draw     (draw) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
          * obs      (obs) int64 8kB 0 1 2 3 4 5 6 7 ... 992 993 994 995 996 997 998 999
        Data variables:
            y        (chain, draw, obs) float32 4MB 2.099 -0.7878 ... -2.089 -2.463
        Attributes:
            created_at:    2026-11-16T19:27:59.891841+00:00
            aimz_version:  0.18.0
            rng_key:       [3288317168, 3869482587]

In [5]:
%watermark -iv

aimz   : 0.18.0
jax    : 0.11.2
logging: 0.5.1.2
numpyro: 0.22.0

